In [11]:
# Add type safety to Topic and ADC classes and the test code

import numpy as np
from typing import List, Optional, Any
from sklearn.metrics.pairwise import cosine_similarity
import warnings

In [ ]:
class Topic:
    def __init__(
        self,
        document_embeddings_hd: np.ndarray,
        centroid_hd: Optional[np.ndarray] = None,
        documents: Optional[List[str]] = None,
        topic_name: Optional[str] = None,
        topic_desc: Optional[str] = None
    ):
        assert isinstance(document_embeddings_hd, np.ndarray), "document_embeddings_hd must be a numpy array"
        if centroid_hd is not None:
            assert isinstance(centroid_hd, np.ndarray), "centroid_hd must be a numpy array"
        if documents is not None:
            assert isinstance(documents, list), "documents must be a list of strings"
        self.document_embeddings_hd = document_embeddings_hd
        self.centroid_hd = centroid_hd
        self.documents = documents or []
        self.topic_name = topic_name
        self.topic_desc = topic_desc

# ADC : Average Document Coherence


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import warnings
from typing import List, Optional, Any


class ADC:
    """
    Average Document Coherence (ADC) metric for topic models.
    """

    def __init__(
        self,
        n_docs: int = -1,  # -1 means all docs in the cluster
        n_intruder_docs: int = 1,
    ):
        assert isinstance(n_docs, int), "n_docs must be an integer"
        assert isinstance(n_intruder_docs, int), "n_intruder_docs must be an integer"
        self.n_docs = n_docs
        self.n_intruder_docs = n_intruder_docs

    def score_one_intr_per_cluster(
        self,
        topic_list: List[Topic],
        random_state: Optional[Any] = None,
    ) -> np.ndarray:
        rng = np.random.default_rng(random_state)
        emb_clusters: List[np.ndarray] = []
        for t in topic_list:
            emb = getattr(t, "document_embeddings_hd", None)
            assert isinstance(emb, np.ndarray), "Topic objects must have document_embeddings_hd field populated as numpy array."
            arr = np.atleast_2d(np.asarray(emb))
            if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
                n_available = arr.shape[0]
                if n_available > self.n_docs:
                    centroid = getattr(t, "centroid_hd", None)
                    if centroid is None:
                        centroid = arr.mean(axis=0)
                    else:
                        centroid = np.asarray(centroid).reshape(-1)
                    sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
                    top_idx = np.argsort(sims)[-self.n_docs:][::-1]
                    arr = arr[top_idx]
            emb_clusters.append(arr)

        scores: List[float] = []
        for i, cluster_emb in enumerate(emb_clusters):
            if cluster_emb.size == 0:
                scores.append(np.nan)
                continue
            other = [np.atleast_2d(c) for j, c in enumerate(emb_clusters) if j != i and c.size > 0]
            if len(other) == 0:
                scores.append(np.nan)
                continue
            other_embs = np.vstack(other)
            intr_idx = int(rng.integers(0, other_embs.shape[0]))
            intr_embedding = other_embs[intr_idx]
            sim = cosine_similarity(intr_embedding.reshape(1, -1), cluster_emb)  # (1, n_docs)
            sim_scaled = (sim + 1) / 2  # Scale to [0, 1]
            scores.append(float(np.mean(sim_scaled)))
        return np.array(scores)

    def score_per_cluster(self, topic_list: List[Topic]) -> dict:
        score_lis: List[np.ndarray] = []
        for _ in range(self.n_intruder_docs):
            score_per_cluster = self.score_one_intr_per_cluster(
                topic_list
            )
            score_lis.append(score_per_cluster)
        res = np.vstack(score_lis).T
        mean_scores = np.mean(res, axis=1)
        ntopics = len(topic_list)
        results: dict = {}
        for k in range(ntopics):
            preview = ""
            t = topic_list[k]
            if getattr(t, "documents", None):
                preview = " - " + str(t.documents[0])[:40].replace("\n", " ").strip()
            label = f"cluster_{k}{preview}"
            results[label] = float(np.round(mean_scores[k], 5))
        return results

    def score(self, topics: List[Topic]) -> float:
        scores = list(self.score_per_cluster(topics).values())
        if all(np.isnan(scores)):
            warnings.warn("ADC: All clusters returned NaN (no valid intruder comparisons possible).")
            return np.nan
        return float(np.nanmean(scores))

In [14]:
np.random.seed(42)
topic1 = Topic(document_embeddings_hd=np.random.randn(4, 8), documents=["doc1", "doc2", "doc3", "doc4"])
topic2 = Topic(document_embeddings_hd=np.random.randn(3, 8), documents=["doc5", "doc6", "doc7"])
topic3 = Topic(document_embeddings_hd=np.random.randn(5, 8), documents=["doc8", "doc9", "doc10", "doc11", "doc12"])

topic_list: List[Topic] = [topic1, topic2, topic3]

adc = ADC(n_docs=-1, n_intruder_docs=2)
result = adc.score(topic_list)
print("ADC score on dummy data:", result)

ADC score on dummy data: 0.49900999999999995


# ADS: Average Document Similarity metric

In [16]:
import numpy as np
import re
from typing import List, Optional, Any
from sklearn.metrics.pairwise import cosine_similarity

class ADS:
    """
    Average Description Similarity (ADS) metric for topic models.
    The average cosine similarity of the embedding of each document d
    to the embedding of the description of the topic assigned to d.
    """

    def __init__(self, n_docs: int = -1, embedder: Optional[Any] = None):
        """
        Args:
            n_docs (int): Number of documents per topic to use (-1 means all).
            embedder: An object with a .get_embeddings(list_of_texts) method returning a dict with 'embeddings' key.
        """
        self.n_docs = n_docs
        self.embedder = embedder

    def get_info(self) -> dict:
        """
        Get information about the metric.
        """
        info = {
            "metric_name": "Average Description Similarity (ADS)",
            "n_docs": self.n_docs,
            "metric_range": "0 to 1, higher is better",
            "description": "The average cosine similarity (scaled to [0,1]) between the embedding of each document and the embedding of its topic description.",
        }
        return info

    def score(self, topics: List[Any]) -> float:
        """
        Args:
            topics: List of Topic-like objects, each with:
                - topic_description (str)
                - document_embeddings_hd (np.ndarray)
                - centroid_hd (optional, np.ndarray)
        Returns:
            float: The average ADS score across all topics.
        """
        assert isinstance(topics, (list, tuple)), "topics must be a list or tuple of Topic objects"
        assert self.embedder is not None, "embedder must be provided"

        # Clean the descriptions
        descriptions = [self.clean_description(getattr(t, "topic_description", "")) for t in topics]

        # Embed the topic descriptions
        topic_desc_embeddings = self.embedder.get_embeddings(descriptions)["embeddings"]

        # Embed the documents in each topic
        emb_clusters: List[np.ndarray] = []
        for t in topics:
            emb = getattr(t, "document_embeddings_hd", None)
            assert emb is not None, "Each topic must have 'document_embeddings_hd'"
            arr = np.atleast_2d(np.asarray(emb))
            # If n_docs > 0, select the most representative documents (top-k by similarity to centroid)
            if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
                n_available = arr.shape[0]
                n_select = min(self.n_docs, n_available)
                centroid = getattr(t, "centroid_hd", None)
                if centroid is None:
                    centroid = arr.mean(axis=0)
                else:
                    centroid = np.asarray(centroid).reshape(-1)
                sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
                top_idx = np.argsort(sims)[-n_select:][::-1]
                arr = arr[top_idx]
            emb_clusters.append(arr)

        similarity_scores: List[float] = []
        for i in range(len(emb_clusters)):
            desc = topic_desc_embeddings[i].reshape(1, -1)
            docs = emb_clusters[i]
            if docs.size == 0:
                similarity_scores.append(np.nan)
                continue
            sims = cosine_similarity(desc, docs)  # Shape: (1, n_docs)
            sims_01 = (sims + 1) / 2  # Now in [0, 1]
            similarity_scores.append(np.nanmean(sims_01))  # Average similarity for the topic

        if len(similarity_scores) == 0 or np.all(np.isnan(similarity_scores)):
            return np.nan
        return round(np.nanmean(similarity_scores), 4)

    @staticmethod
    def clean_description(desc: str) -> str:
        if not desc:
            return ""
        desc = desc.strip().strip('\'"')
        desc = re.sub(r'(\*\*|\*|`)+', '', desc)
        desc = re.sub(r'^#{1,6}\s*', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'^\s*[\-\*\+]\s+', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'^\s*\d+\.\s+', '', desc, flags=re.MULTILINE)
        desc = re.sub(r'\s+', ' ', desc)
        return desc.strip()

In [17]:
# Dummy data and test code for standalone ADS metric

import numpy as np
from typing import List, Optional, Any

# Dummy embedder class for testing
class DummyEmbedder:
    def get_embeddings(self, texts: List[str]) -> dict:
        # For each text, return a random embedding of size 8
        np.random.seed(42)  # For reproducibility
        embeddings = np.random.randn(len(texts), 8)
        return {"embeddings": embeddings}

# Dummy Topic class for testing
class DummyTopic:
    def __init__(
        self,
        topic_description: str,
        document_embeddings_hd: np.ndarray,
        centroid_hd: Optional[np.ndarray] = None
    ):
        self.topic_description = topic_description
        self.document_embeddings_hd = document_embeddings_hd
        self.centroid_hd = centroid_hd

# Create dummy topics
np.random.seed(1)
topic1 = DummyTopic(
    topic_description="Sports news and events",
    document_embeddings_hd=np.random.randn(4, 8)
)
topic2 = DummyTopic(
    topic_description="Technology and gadgets",
    document_embeddings_hd=np.random.randn(3, 8)
)
topic3 = DummyTopic(
    topic_description="Health and wellness tips",
    document_embeddings_hd=np.random.randn(5, 8)
)

topic_list: List[DummyTopic] = [topic1, topic2, topic3]

# Instantiate ADS metric with dummy embedder
ads = ADS(n_docs=-1, embedder=DummyEmbedder())

# Run the metric
result = ads.score(topic_list)
print("ADS score on dummy data:", result)

ADS score on dummy data: 0.441
